## Unit 12. 程序最適化 課堂作業
- 課程編號: ChemE-3502
- 學年度: 114下
- 上課時間: 每週四 09:00-12:00
-
- 指導教授: ＯＯＯ 教授
- 學生姓名: ＯＯＯ
- 學號: m12345678
- email帳號: fcu.m12345678@gmail.com

---
### 0. 環境設定

In [ ]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'Unit12_Homework'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3502'
    os.remove(shortcut_path)
except (FileNotFoundError, OSError):
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3502')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Unit12'
        OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        FIG_DIR      = OUTPUT_DIR / 'figs'
    else:
        print("⚠️ 找不到雲端 ChemE-3502 路徑，請確認資料夾名稱是否正確")
else:
    NOTEBOOK_DIR = Path.cwd()
    OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    FIG_DIR      = OUTPUT_DIR / 'figs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

---
### 1. 載入套件

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy.optimize import minimize_scalar, minimize, linprog, differential_evolution

# 繪圖樣式設定
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'lines.linewidth': 2,
    'axes.unicode_minus': False,
})

print("✓ 套件載入完成")
print(f"  numpy  版本: {np.__version__}")
import scipy
print(f"  scipy  版本: {scipy.__version__}")
import matplotlib
print(f"  matplotlib 版本: {matplotlib.__version__}")

---
## I. 主要作業：程序最適化三題練習

本作業共三道題目，涵蓋**單變數最適化**、**線性規劃**與**非線性約束最適化**三種類型。請依序完成各題，並在程式碼中加入必要的中文或英文說明。

| 題號 | 主題 | 使用工具 | 分數比重 |
|------|------|----------|----------|
| 題 I | 過濾實驗所需最長過濾時間 | `minimize_scalar` | 30% |
| 題 II | 石化精煉最適化 | `linprog` | 35% |
| 題 III | 金屬切割最適化 | `minimize` (SLSQP) | 35% |

---
### **題 I: 單變數最適化 — 過濾實驗所需最長過濾時間**
> 改編自教材 ch6 習題 6-3-3

#### 問題描述

以定速進行過濾實驗，其過濾所需之時間 $t_f$ 與壓差 $\Delta P_c$ 、過濾面積 $A$ 、液體黏度 $\mu$ 及濾餅操作性質之間關係如下：

$$
t_f = \beta \frac{\Delta P_c A^2}{\mu M^2 C} \, x_c \exp(-a x_c + b)
$$

其中 $x_c$ 為濾餅中固體質量分率，各參數數值如下：

| 參數 | 物理意義 | 數值 | 單位 |
|------|---------|------|------|
| $\Delta P_c$ | 濾餅兩端壓差 | 20 | psig |
| $A$ | 過濾面積 | 250 | ft² |
| $\mu$ | 濾液黏度 | 20 | c.p. |
| $M$ | 濾液質量流率 | 75 | lb/min |
| $C$ | 每磅濾液中所含固體量 | 0.01 | lb/lb |
| $a$ | 模式參數 | 3.643 | — |
| $b$ | 模式參數 | 2.68 | — |
| $\beta$ | 模式參數 | $3.2 \times 10^{-8}$ | $(lb/ft)^2$ |

**目標**：找出使過濾時間 $t_f$ **最長**的最佳固體質量分率 $x_c^*$，以及對應的最長過濾時間。

> 提示：最大化 $t_f(x_c)$ 等價於最小化 $-t_f(x_c)$，請使用 `scipy.optimize.minimize_scalar(method='bounded')` 求解，搜尋範圍為 $x_c \in [0.1, 0.8]$。

#### 作業要求
1. 定義目標函數 $-t_f(x_c)$（用於最小化）
2. 使用 `minimize_scalar` 求解最佳 $x_c^*$ 與最長 $t_f^*$
3. 繪製 $t_f$ vs $x_c$ 的曲線圖，標示最佳點位置
4. 列印最佳結果

> 參考答案：最適固體質量分率 $\approx 27.45\%$，最長過濾時間 $\approx 673.2$ min

In [ ]:
# ========================================
# 題 I: 過濾實驗所需最長過濾時間
# ========================================

# --- 參數定義 ---
delta_Pc = 20    # psig
A_area   = 250   # ft^2
mu       = 20    # c.p.
M        = 75    # lb/min
C        = 0.01  # lb/lb
a        = 3.643
b        = 2.68
beta     = 3.2e-8  # (lb/ft)^2

# --- (1) 定義目標函數 (最大化 t_f = 最小化 -t_f) ---
# TODO: 請在此定義目標函數


# --- (2) 使用 minimize_scalar 求解 ---
# TODO: 請在此求解並列印結果


# --- (3) 繪製 t_f vs x_c 曲線圖 ---
# TODO: 請繪製目標函數曲線，並標示最佳點位置


---
### **題 II: 線性規劃 — 石化精煉最適化**
> 改編自教材 ch6 習題 6-3-1

#### 問題描述

某一石化精煉廠由兩種原料成分生產兩種產品。考量其獲利及規格配方等限制要求，可形成以下之**線性規劃問題**：

$$
\max_{y_i} \; J = 6y_1 + y_2 + 4y_3 - y_4
$$

受制於以下限制條件：

$$
y_1 + y_3 \leq 1000
$$

$$
y_2 + y_4 \leq 2000
$$

$$
0.65y_1 - 0.35y_2 \leq 0
$$

$$
0.70y_1 - 0.30y_2 \leq 0
$$

$$
0.40y_3 - 0.60y_4 \leq 0
$$

$$
0.30y_3 - 0.70y_4 \leq 0
$$

$$
y_1, y_2, y_3, y_4 \geq 0
$$

**目標**：決定最適條件 $y_1, y_2, y_3, y_4$ 各多少，此時最大獲利 $J^*$ 為多少？

> 提示：最大化 $J$ 等價於最小化 $-J$，請使用 `scipy.optimize.linprog()` 求解，注意目標係數需取**負號**。

#### 作業要求
1. 建立目標函數係數向量（最小化 $-J$）
2. 建立不等式限制條件矩陣 $A_{ub}$ 及右側向量 $b_{ub}$
3. 設定變數下限（$y_i \geq 0$）
4. 呼叫 `linprog()` 求解，解讀並列印最適結果
5. 繪製最適解的長條圖，顯示各變數數值

> 參考答案： $y_1^* = 800$、 $y_2^* = 1866.67$、 $y_3^* = 200$、 $y_4^* = 133.33$，最大獲利 $J^* = 7333.33$

In [ ]:
# ========================================
# 題 II: 石化精煉最適化 (線性規劃)
# ========================================

# --- (1) 目標函數係數 (最小化 -J = -6y1 - y2 - 4y3 + y4) ---
# TODO: 請在此定義 c 向量


# --- (2) 不等式限制矩陣 A_ub x <= b_ub ---
# TODO: 請在此定義 A_ub 和 b_ub


# --- (3) 變數邊界 (yi >= 0) ---
# TODO: 請在此定義 bounds


# --- (4) 求解 ---
# TODO: 請呼叫 linprog() 並列印求解結果


# --- (5) 繪製最適解長條圖 ---
# TODO: 繪製 y1, y2, y3, y4 的長條圖


---
### **題 III: 非線性約束最適化 — 金屬切割最適化**
> 改編自教材 ch6 習題 6-3-2

#### 問題描述

一個金屬切割之最佳化問題，目標為最小化切割成本函數：

$$
\min_{x_1, x_2} \; f = \frac{982}{x_1 x_2} + 8.1 \times 10^{-9} \, x_1^{1.222} \, x_2^{2.333}
$$

受制於以下限制條件：

$$
x_1^{0.78} x_2 \leq 4 \times 10^3 \quad \text{(不等式限制 1)}
$$

$$
x_1 x_2^2 \geq 3.8 \times 10^5 \quad \text{(不等式限制 2)}
$$

$$
1 \leq x_1 \leq 35, \quad 49.1 \leq x_2 \leq 982 \quad \text{(變數邊界)}
$$

其中 $x_1$ 為切削速度，$x_2$ 為進給量。

> 提示：
> - 使用 `scipy.optimize.minimize(method='SLSQP')` 求解
> - 不等式限制需轉換為 SciPy 格式：$c(\mathbf{x}) \geq 0$
>   - 限制 1: $4000 - x_1^{0.78} x_2 \geq 0$
>   - 限制 2: $x_1 x_2^2 - 380000 \geq 0$
> - 可嘗試多組起始猜測值，比較收斂結果

#### 作業要求
1. 定義目標函數 $f(x_1, x_2)$
2. 定義兩個不等式限制條件（SciPy 格式 $c \geq 0$）
3. 設定變數邊界 `bounds`
4. 使用 `minimize(method='SLSQP')` 求解，解讀最適結果
5. 驗證所有限制條件是否滿足（殘差檢查）
6. 繪製可行區域輪廓圖，標示最適解位置

> 參考答案： $x_1^* = 35$，$x_2^* \approx 153.34$，$f^* \approx 0.2614$

In [ ]:
# ========================================
# 題 III: 金屬切割最適化 (非線性約束最適化)
# ========================================

# --- (1) 定義目標函數 ---
# TODO: 請在此定義 objective(x)


# --- (2) 定義不等式限制條件 (SciPy 格式: c(x) >= 0) ---
# TODO: 限制 1: 4000 - x1^0.78 * x2 >= 0
# TODO: 限制 2: x1 * x2^2 - 380000 >= 0
# constraints = [...]


# --- (3) 設定變數邊界 ---
# x1 in [1, 35], x2 in [49.1, 982]
# TODO: 定義 bounds


# --- (4) 設定起始猜測值並求解 ---
# TODO: 嘗試不同的初始值 x0，呼叫 minimize(method='SLSQP') 求解


# --- (5) 驗證限制條件滿足度 ---
# TODO: 列印各限制條件殘差，確認是否 >= 0


# --- (6) 繪製可行區域輪廓圖與最適解位置 ---
# TODO: 繪製等高線圖，標示最適解 (x1*, x2*)


---
## II. 額外加分作業 (自由繳交)
- 學生可自由選擇是否繳交加分作業。(下週上課前完成 email 電子檔)
- 分數會加在最後總成績上，每份作業加 0.1 ~ 1.0 分。
- 分數加的不多，真的不一定要交加分作業，正常出席上課做好期末報告即可。
- 加分作業不一定要全部都做完，想繳交至少完成其中一項即可。
- 請務必自行完成！你可以問 AI、問同學、問學長姊，但一定要自行**吸收、執行、整理**結果！
- 不要貼 AI 的答案寄給老師看，你自己看就好。
- 如果系統自動比對發現作業內容（與他人雷同、原創性低、抄襲比例過高），作業的分數會 0 分。
- 如果你直接 100% 複製別人的作業繳交，你會直接被當掉（提供作業給別人複製的也一樣）。
- 老師看你作業要花時間，真的有心想寫加分作業，請好好自己做。

---
### **加分題 1: 蒸餾塔分離程序最佳獲利**
> 改編自教材 ch6 習題 6-3-6

#### 問題描述

考慮一個蒸餾塔之分離程序，其中 $x_1 \sim x_5$ 各表示如圖中該處管路之流率（ $\mathrm{bbl/day}$ ）：
- light end、medium solvent 和 heavy solvent 之售價分別為 $53$、$68$、$42\ \$/\mathrm{bbl}$
- 各蒸餾塔處理之成本為 $1.25\ \$/\mathrm{bbl}$ 進料，原料成本為 $42\ \$/\mathrm{bbl}$

**目標函數（獲利最大化）：**

$$
\max \; f = 53x_3 + 68x_4 + 42x_5 - 42x_1 - 1.25x_1 - 1.25x_2
$$

**等式限制（質量平衡）：**

$$
x_1 = x_3 + x_4 + x_5, \quad 0.6x_1 = x_4 + x_5, \quad x_3 = 0.4x_1, \quad x_2 = 0.6x_1
$$

**不等式限制與邊界：**

$$
200 \leq x_4 \leq 400, \quad x_4 \geq 0.5x_2, \quad x_5 \geq 0.3x_2, \quad x_1 \leq 2000, \quad x_i \geq 0
$$

> 提示：本題為線性規劃問題，可用 `scipy.optimize.linprog()` 求解。等式限制可由質量平衡方程式化簡後納入 `A_eq` 及 `b_eq` 參數。

#### 加分要求
1. 建立完整 LP 數學模型（目標函數、等式/不等式限制、邊界）
2. 使用 `linprog()` 求解，解讀並列印各流率最適值與最大獲利
3. 進行敏感度分析：探討 medium solvent 售價從 60 到 80 $/bbl 變化時，最大獲利的變化

> 參考答案： $\mathbf{x}^* = [1333.33,\ 800,\ 533.33,\ 400,\ 400]$，$f^* = 13600$

In [ ]:
# ========================================
# 加分題 1: 蒸餾塔分離程序最佳獲利
# ========================================

# TODO: 請在此完成蒸餾塔 LP 最適化求解
# 提示: 變數順序建議為 [x1, x2, x3, x4, x5]


---
### **加分題 2: 平行競爭反應之最適溫度規劃 (動態最適化)**
> 改編自教材 ch6 習題 6-3-17

#### 問題描述

一個平行競爭反應（Parallel reaction）之最適溫度規劃問題，目標為：

$$
\max_{T(t)} \; C_B(t_f)
$$

受制於以下動態方程式（ODE 限制）：

$$
\frac{dC_A}{dt} = -k_1 C_A, \quad C_A(0) = 1
$$

$$
\frac{dC_B}{dt} = k_1 C_A - k_2 C_B, \quad C_B(0) = 0
$$

其中反應速率常數：

$$
k_i = k_{i0} \exp\!\left(-\frac{E_i}{RT}\right), \quad i = 1, 2
$$

| 參數 | 數值 |
|------|------|
| $k_{10}$ | $65.6\ \mathrm{s}^{-1}$ |
| $k_{20}$ | $1970\ \mathrm{s}^{-1}$ |
| $E_1$ | $1.0 \times 10^4\ \mathrm{cal/mol}$ |
| $E_2$ | $1.6 \times 10^4\ \mathrm{cal/mol}$ |
| $R$ | $1.987\ \mathrm{cal/(mol \cdot K)}$ |
| $t_f$ | $12.5\ \mathrm{s}$ |

**最適化策略**：
將反應時間 $[0, t_f]$ 分割為 $n$ 段，假設各段溫度為常數，以各段溫度 $T_1, T_2, \ldots, T_n$ 作為決策變數，溫度操作範圍為 $298 \leq T \leq 800\ \mathrm{K}$。

#### 加分要求
1. 撰寫 ODE 反應動態方程式（使用 `scipy.integrate.solve_ivp()`）
2. 建立目標函數（對給定溫度分布求解 ODE，回傳 $-C_B(t_f)$）
3. 使用 `minimize(method='SLSQP')` 搭配邊界限制求解最佳溫度分布
4. 繪製最佳操作溫度分布圖（階梯圖）與濃度 $C_A(t)$、$C_B(t)$ 隨時間變化圖
5. 探討分割段數 $n = 5, 10, 20$ 對最佳解的影響

> 參考答案：反應終了時最高 $C_B$ 濃度約為 $0.4787$

In [ ]:
# ========================================
# 加分題 2: 平行競爭反應最適溫度 (動態最適化)
# ========================================
from scipy.integrate import solve_ivp

# --- 參數 ---
k10 = 65.6      # s^-1
k20 = 1970.0    # s^-1
E1  = 1.0e4     # cal/mol
E2  = 1.6e4     # cal/mol
R   = 1.987     # cal/(mol·K)
tf  = 12.5      # s
n_segments = 10  # 時間分割段數 (可調整)

# TODO: 請在此完成動態最適化求解
# 提示:
# 1. 撰寫 ode_func(t, y, T)
# 2. 撰寫 objective(T_vec) - 求解 ODE 並回傳 -CB(tf)
# 3. 設定初始猜測值 T0 和邊界 bounds
# 4. 呼叫 minimize(method='SLSQP')


---
### **加分題 3: 自由實驗 — 全域最適化方法比較**

選擇任一 ch6 習題（如 6-3-7 反應器組設計、6-3-12 連續式反應器串列系統等），分別使用以下至少兩種方法求解同一問題，並進行比較分析：

1. 局部最適化：`scipy.optimize.minimize(method='SLSQP')`（多組不同起始猜測值）
2. 全域最適化之一：`scipy.optimize.differential_evolution()`
3. 全域最適化之二（選填）：`scipy.optimize.dual_annealing()` 或 `scipy.optimize.basinhopping()`

#### 加分要求
- 清楚描述所選擇之問題（目標函數、限制條件）
- 實作並執行至少兩種方法
- 列製比較表格（目標函數值、計算時間、收斂結果）
- 分析：局部最適化是否容易陷入局部最小值？全域最適化是否能找到更好的解？
- 討論計算效率與解的品質之取捨

In [ ]:
# ========================================
# 加分題 3: 全域最適化方法比較
# ========================================

# TODO: 請在此完成全域最適化方法比較實驗
# 提示: 可使用 differential_evolution 或 dual_annealing


---
## 💭 思考題

請在以下空格中簡短回答（可以文字說明，不需要程式碼）：

1. `minimize_scalar` 與 `minimize` 的主要差異是什麼？何時應選擇 `minimize_scalar`？
2. 線性規劃（LP）求解時，為何最適解通常出現在**可行區域的頂點**（vertex）上？
3. 在使用 `linprog()` 時，為何最大化問題需要將目標函數係數**取負號**？
4. SciPy 的不等式限制約定（ $c(\mathbf{x}) \geq 0$ ）與 MATLAB 的 fmincon（ $c(\mathbf{x}) \leq 0$ ）方向相反，請說明如何轉換。
5. 使用 `minimize(method='SLSQP')` 求解非線性約束問題時，起始猜測值 `x0` 為何很重要？如何選擇一個好的 `x0`？
6. 全域最適化方法（如 `differential_evolution`）如何避免局部最小值問題？其代價為何？

*（在此填寫你的回答）*

---
# 想對老師說的話